In [1]:
import os
import pandas as pd

print(os.getcwd())

/Users/evannuss/Documents/retail_analytics_portfolio/notebooks


In [2]:
import pandas as pd

customers = pd.read_csv("../data/raw/customers.csv")
employees = pd.read_csv("../data/raw/employees.csv")
order_items = pd.read_csv("../data/raw/order_items.csv")
orders = pd.read_csv("../data/raw/orders.csv")
products = pd.read_csv("../data/raw/products.csv")
returns = pd.read_csv("../data/raw/returns.csv")
stores = pd.read_csv("../data/raw/stores.csv")


In [3]:
order_id_duplicates = orders.duplicated(subset="order_id", keep=False)

order_id_duplicates_rows = orders[order_id_duplicates]

print(order_id_duplicates_rows)


      order_id customer_id           order_date store_id employee_id  \
588    O000589      C06384  2025-03-28 04:00:15     S012       E0121   
1163   O001164      C04233  2025-09-07 18:58:30     S009       E0264   
1231   O001232      C07371  2024-10-31 20:42:45     S005         NaN   
1571   O001572      C03195  2024-04-15 01:13:40     S010         NaN   
3556   O003557      C07607  2025-11-19 06:02:00     S019       E0079   
...        ...         ...                  ...      ...         ...   
50095  O042573      C04232  2024-04-14 13:43:38     S021         NaN   
50096  O045107      C06483  2024-07-06 05:19:10     S005         NaN   
50097  O028133      C06605  2025-09-21 21:00:50     S025         NaN   
50098  O041453      C02652  2026-02-09 05:13:43     S013       E0077   
50099  O027747      C02957  2025-03-10 11:51:18     S015       E0130   

          channel payment_method order_status  
588    Mobile App      Gift Card    Completed  
1163        Store         PayPal    Com

In [4]:
print("Count: ", order_id_duplicates_rows.count())

Count:  order_id          200
customer_id       200
order_date        200
store_id          200
employee_id       118
channel           200
payment_method    200
order_status      200
dtype: int64


# Data Quality Assessment

# Customers table overview

In [6]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 8120 entries, 0 to 8119
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   customer_id    8120 non-null   str  
 1   first_name     8120 non-null   str  
 2   last_name      8120 non-null   str  
 3   email          7770 non-null   str  
 4   phone          7870 non-null   str  
 5   city           8120 non-null   str  
 6   state          8120 non-null   str  
 7   region         8120 non-null   str  
 8   customer_tier  8120 non-null   str  
 9   signup_date    8120 non-null   str  
dtypes: str(10)
memory usage: 634.5 KB


In [12]:
customers.head()


,customer_id,first_name,last_name,email,phone,city,state,region,customer_tier,signup_date
0,C00001,Michael,Robinson,michael.robinson1@example.com,555-496-7825,Denver,CO,West,Gold,2025-02-04
1,C00002,Michael,Smith,michael.smith2@example.com,555-307-8259,Atlanta,GA,South,Silver,2025-07-21
2,C00003,Sophia,Harris,sophia.harris3@example.com,555-659-4218,San Francisco,CA,West,Silver,2023-08-26
3,C00004,Amelia,Garcia,amelia.garcia4@example.com,555-291-3847,Philadelphia,PA,Northeast,Standard,2024-02-12
4,C00005,Robert,Wilson,robert.wilson5@example.com,555-669-5329,Atlanta,GA,South,Silver,2022-11-26


In [23]:
# Number of Customers by region
customers_by_region = customers.groupby("region")["customer_id"].count()

print(customers_by_region)

region
Midwest      1241
Northeast    1203
South        2863
West         2813
Name: customer_id, dtype: int64


# Employees

In [ ]:
employees.info()

# hire_date is a string -- we will need to change that downstream
# An employee is tied to a specific store_id
# evaluate if any employees belong to more than one store

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   employee_id  300 non-null    str  
 1   first_name   300 non-null    str  
 2   last_name    300 non-null    str  
 3   role         300 non-null    str  
 4   store_id     300 non-null    str  
 5   city         300 non-null    str  
 6   state        300 non-null    str  
 7   hire_date    300 non-null    str  
dtypes: str(8)
memory usage: 18.9 KB


In [ ]:
employees.head()

,employee_id,first_name,last_name,role,store_id,city,state,hire_date
0,E0001,Sofia,Thompson,Sales Lead,S005,Seattle,WA,2022-11-13
1,E0002,Evelyn,Brown,Assistant Manager,S014,New York,NY,2019-12-18
2,E0003,Christopher,Lopez,Assistant Manager,S011,Chicago,IL,2024-02-09
3,E0004,Grace,Thomas,Assistant Manager,S016,Dallas,TX,2020-02-04
4,E0005,Amelia,Hernandez,Sales Associate,S010,Seattle,WA,2019-07-03


In [28]:
employee_by_store = employees.groupby(["employee_id", "store_id"])["store_id"].count()
employees
print(employee_by_store)

# Found that some employees (E0298) are not assigned to a store_id
# I am curious of two things: the hire date of the employees (could it be relatively recent where they have not been assigned)
# Or what their role is -- if they are corporate and/or interns, they may not be assigned to a single store
print("--------------------------")

employees_w_unknown_store = employees[employees["store_id"] == "UNKNOWN"]
print(employees_w_unknown_store)

employee_id  store_id
E0001        S005        1
E0002        S014        1
E0003        S011        1
E0004        S016        1
E0005        S010        1
                        ..
E0296        S022        1
E0297        UNKNOWN     1
E0298        UNKNOWN     1
E0299        S004        1
E0300        S017        1
Name: store_id, Length: 300, dtype: int64
--------------------------
    employee_id first_name  last_name               role store_id  \
28        E0029     Sophia      Moore    Sales Associate  UNKNOWN   
61        E0062      Brian      Moore         sales lead  UNKNOWN   
72        E0073       Ella   Robinson      Store Manager  UNKNOWN   
98        E0099      Grace     Wilson  Assistant Manager  UNKNOWN   
109       E0110    Michael      Perez  Assistant Manager  UNKNOWN   
119       E0120     Hannah     Martin         Sales Lead  UNKNOWN   
132       E0133    Abigail      Moore  Assistant Manager  UNKNOWN   
164       E0165     Joshua      Lopez   Customer Service  UN

# Found no correlation to UNKNOWN store_id of my two hypotheses above
# I want analyze the list of all job titles (data quality check)

In [ ]:
role_list = employees.role
print(role_list.drop_duplicates())

# Found the naming inconsistencies
# Will address with standardization

0            Sales Lead
1     Assistant Manager
4       Sales Associate
6         Store Manager
7      Customer Service
8      customer service
12      sales associate
17        store manager
27           sales lead
62    assistant manager
Name: role, dtype: str


# Order Items

In [ ]:
order_items.info()
# Might convert Dtypes to be decimal (int64/float64 variables) since we are calculating a financial value (line_revenue)

<class 'pandas.DataFrame'>
RangeIndex: 123699 entries, 0 to 123698
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      123699 non-null  str    
 1   product_id    123699 non-null  str    
 2   quantity      123699 non-null  int64  
 3   unit_price    123699 non-null  float64
 4   discount_pct  123699 non-null  float64
 5   line_revenue  123699 non-null  float64
dtypes: float64(3), int64(1), str(2)
memory usage: 5.7 MB


In [ ]:
order_items.head()

# Want to look for duplicate order_id/product_id combinations
# a single order_id should NOT have more than one line for the same product_id

orders_w_duplicate_product_ids = order_items.duplicated(subset=["order_id", "product_id"], keep=False)

final_list = order_items[orders_w_duplicate_product_ids]
print(len(final_list))
print("----------------------")

# Want to see if the duplicate lines (1007 count) are different lines
final_list.sort_values(["order_id", "product_id"], ascending=False)

# Found multiple line items where a single order_id, will have multiple records of a single product_id (rather than have an increased quantity)
# I intend on consolidating these, but we could guess that maybe these were orders placed after the initial order submissiong and were given the option to include it in the same order delivery
# (Check the data dictionary for if the above is just a a possible scenario)



1007
----------------------


,order_id,product_id,quantity,unit_price,discount_pct,line_revenue
123257,O049917,P0104,1,474.03,0.20,379.22
123635,O049917,P0104,1,474.03,0.20,379.22
123166,O049878,P0313,1,553.45,0.05,525.78
123450,O049878,P0313,1,553.45,0.05,525.78
122725,O049699,P0353,2,634.10,0.20,1014.56
...,...,...,...,...,...,...
123651,O000146,P0456,4,487.60,0.15,1657.84
233,O000098,P0317,2,212.58,0.10,382.64
123638,O000098,P0317,2,212.58,0.10,382.64
86,O000032,P0045,2,117.04,0.20,187.26
